# 05 - Enriquecimiento con comuna por barrio

## Objetivo
Agregar la columna `comuna` al dataset usando el CSV oficial de barrios de CABA (`barrios_caba.csv`).
Para los barrios que no coinciden exactamente (tildes, sub-barrios, nombres alternativos)
se aplica un mapeo manual.

## Entradas
- `outputs/interm/02_2_df_con_dolar.csv` — dataset principal
- `barrios_caba.csv` — CSV oficial con columnas `nombre` y `comuna`

## Salidas
- `outputs/interm/05_df_con_comuna.csv`
- `outputs/tablas/05_reporte_comuna.csv` — barrios sin comuna asignada


## Librerías y parámetros

In [23]:
from pathlib import Path
import pandas as pd

out_interm = Path('outputs/interm')
out_tablas = Path('outputs/tablas')
out_interm.mkdir(parents=True, exist_ok=True)
out_tablas.mkdir(parents=True, exist_ok=True)

pd.set_option('display.max_rows', None)

## 1. Cargar datasets

In [24]:
df = pd.read_csv(out_interm / '02_2_df_con_dolar.csv')
print(f'Dataset principal: {len(df)} filas')

df_barrios = pd.read_csv('barrios_caba.csv')
print(f'Barrios CABA: {len(df_barrios)} filas')
print(f'Columnas: {df_barrios.columns.tolist()}')
display(df_barrios[['nombre', 'comuna']].head())

Dataset principal: 104084 filas
Barrios CABA: 48 filas
Columnas: ['id', 'objeto', 'nombre', 'comuna', 'perimetro_', 'area_metro', 'geometry']


,nombre,comuna
0,Agronomia,15
1,Almagro,5
2,Balvanera,3
3,Barracas,4
4,Belgrano,13


## 2. Primer mapeo: desde el CSV oficial

In [25]:
# Diccionario nombre_oficial → comuna
barrio_comuna = df_barrios.set_index('nombre')['comuna'].to_dict()

# Primer intento de mapeo
df['comuna'] = df['barrio'].map(barrio_comuna)

n_ok    = df['comuna'].notna().sum()
n_nulo  = df['comuna'].isna().sum()
print(f'Con comuna tras primer mapeo: {n_ok} ({round(100*n_ok/len(df), 1)}%)')
print(f'Sin comuna (requieren mapeo manual): {n_nulo}')

Con comuna tras primer mapeo: 72541 (69.7%)
Sin comuna (requieren mapeo manual): 31543


## 3. Mapeo manual para los que no coinciden

Dos tipos de casos:
- **Nombre distinto** en el CSV oficial (tildes, abreviaturas) → se indica el nombre correcto
- **Sub-barrios** (Palermo Soho, Caballito Norte, etc.) → se asigna la comuna directamente

In [26]:
mapeo_manual = {
    # Nombres distintos en el CSV oficial (valor = nombre en barrios_caba.csv)
    'Núñez':               'Nuñez',
    'Agronomía':           'Agronomia',
    'Villa Pueyrredón':    'Villa Pueyrredon',
    'Constitución':        'Constitucion',
    'Villa General Mitre': 'Villa Gral. Mitre',
    'Villa del Parque': 'Villa Del Parque',  
    'La Paternal':         'Paternal',

    # Sub-barrios y nombres alternativos (valor = número de comuna)
    'Barrio Norte':                    2,
    'Distrito Quartier':               1,
    'Palermo Hollywood':               14,
    'Las Cañitas':                     14,
    'Palermo Chico':                   14,
    'Palermo Soho':                    14,
    'Palermo Nuevo':                   14,
    'Palermo Viejo':                   14,
    'Centro / Microcentro':            1,
    'Belgrano C':                      13,
    'Belgrano R':                      13,
    'Belgrano Chico':                  13,
    'Botánico':                        2,
    'San Nicolás':                     1,
    'Congreso':                        3,
    'Caballito Sur':                   6,
    'Caballito Norte':                 6,
    'Once':                            3,
    'Abasto':                          3,
    'Flores Norte':                    7,
    'Flores Sur':                      7,
    'Almagro Norte':                   5,
    'Almagro Sur':                     5,
    'Tribunales':                      1,
    'Parque Centenario':               6,
    'Lomas de Núñez':                  13,
    'Parque Rivadavia':                7,
    'Cid Campeador':                   5,
    'Barrio Chino':                    13,
    'Primera Junta':                   6,
    'Floresta Norte':                  10,
    'Floresta Sur':                    10,
    'Puerto Retiro':                   1,
    'Barrio Parque':                   13,
    'Catalinas':                       1,
    'Barrio Parque General Belgrano':  13,
    'Pompeya':                         4,
    'Naón':                            9,

    # Sin comuna asignable (fuera de CABA o genéricos)
    'Otro':              None,
    'Temperley':         None,
    'Los Perales':       None
}

def asignar_comuna(barrio, comuna_actual):
    if pd.notna(comuna_actual):
        return comuna_actual          # ya tiene comuna, no tocar

    correccion = mapeo_manual.get(barrio)

    if correccion is None:
        return None                   # sin mapeo disponible
    elif isinstance(correccion, int):
        return correccion             # comuna directa
    else:
        return barrio_comuna.get(correccion, None)  # buscar nombre alternativo

df['comuna'] = df.apply(
    lambda row: asignar_comuna(row['barrio'], row['comuna']),
    axis=1
)

n_ok   = df['comuna'].notna().sum()
n_nulo = df['comuna'].isna().sum()
print(f'Con comuna tras mapeo manual: {n_ok} ({round(100*n_ok/len(df), 1)}%)')
print(f'Sin comuna (NaN):             {n_nulo}')

Con comuna tras mapeo manual: 104046 (100.0%)
Sin comuna (NaN):             38


## 4. Verificar resultado

In [27]:
print('Barrios sin comuna asignada:')
sin_comuna = df[df['comuna'].isna()]['barrio'].value_counts()
print(sin_comuna if len(sin_comuna) else 'Ninguno ✓')

print('\nAvisos por comuna:')
display(
    df.groupby('comuna')
    .size()
    .reset_index(name='cantidad_avisos')
    .sort_values('comuna')
)

Barrios sin comuna asignada:
barrio
Otro           36
Los Perales     1
Temperley       1
Name: count, dtype: int64

Avisos por comuna:


,comuna,cantidad_avisos
0,1.0,11422
1,2.0,15152
2,3.0,4932
3,4.0,1822
4,5.0,4312
5,6.0,6718
6,7.0,3347
7,8.0,263
8,9.0,887
9,10.0,1849


## 5. Exportar

In [29]:
# Dataset enriquecido
out_path = out_interm / '02_3_df_con_comuna.csv'
df['comuna'] = df['comuna'].astype('Int64')
df.to_csv(out_path, index=False)


print(f'Guardado: {out_path}')
print(f'Filas: {len(df)} | Columnas: {df.shape[1]}')

Guardado: outputs/interm/02_3_df_con_comuna.csv
Filas: 104084 | Columnas: 39
